Unzipping files from the US Drought Monitor and creating year-level shapefiles

In [ ]:
import os
import zipfile

import geopandas as gpd
import pandas as pd

In [ ]:
def unzip_files(dir_path: str, delete_zip: bool = True):
    """Unzips all zip files in a given directory
    Args
        dir_path: path to your directory with zip files in it
        delete_zip: if true, delete the zip file after unzipping
    Returns
        None
    """
    for file in os.listdir(dir_path):
        if file.endswith(".zip"):
            file_path = os.path.join(dir_path, file)
            out_path = os.path.join(dir_path, os.path.splitext(file)[0])

            with zipfile.ZipFile(file_path, "r") as zip_ref:
                zip_ref.extractall(out_path)  # unzip files to a folder w/ the same name
                print(f"Extracted {file} to {out_path}")
            if delete_zip:
                os.remove(file_path)
                print(f"deleting {file}")


unzip_files(
    "/Users/kennyalexlin/Desktop/MIDS/W209/W209-Viz-Drought-in-CA/data/drought_monitor/shapefiles_zipped/2019_USDM_M"
)

Extracted USDM_20191224_M.zip to /Users/kennyalexlin/Desktop/MIDS/W209/W209-Viz-Drought-in-CA/data/drought_monitor/shapefiles_zipped/2019_USDM_M/USDM_20191224_M
deleting USDM_20191224_M.zip
Extracted USDM_20190219_M.zip to /Users/kennyalexlin/Desktop/MIDS/W209/W209-Viz-Drought-in-CA/data/drought_monitor/shapefiles_zipped/2019_USDM_M/USDM_20190219_M
deleting USDM_20190219_M.zip
Extracted USDM_20190611_M.zip to /Users/kennyalexlin/Desktop/MIDS/W209/W209-Viz-Drought-in-CA/data/drought_monitor/shapefiles_zipped/2019_USDM_M/USDM_20190611_M
deleting USDM_20190611_M.zip
Extracted USDM_20190827_M.zip to /Users/kennyalexlin/Desktop/MIDS/W209/W209-Viz-Drought-in-CA/data/drought_monitor/shapefiles_zipped/2019_USDM_M/USDM_20190827_M
deleting USDM_20190827_M.zip
Extracted USDM_20190430_M.zip to /Users/kennyalexlin/Desktop/MIDS/W209/W209-Viz-Drought-in-CA/data/drought_monitor/shapefiles_zipped/2019_USDM_M/USDM_20190430_M
deleting USDM_20190430_M.zip
Extracted USDM_20190101_M.zip to /Users/kennyalexl

In [ ]:
YEAR = 2019

dir_path = f"/Users/kennyalexlin/Desktop/MIDS/W209/W209-Viz-Drought-in-CA/data/drought_monitor/shapefiles_zipped/{YEAR}_USDM_M"
months = []
df = gpd.GeoDataFrame()
for i in sorted(os.listdir(dir_path)):
    # use the first reading in each month as the month's drought reading
    # drought readings are once a week, but this is probably a decent approximation
    date_str = i.split("_")[1]
    month = date_str[4:6]
    year = date_str[:4]
    if month not in months:
        months.append(month)
        file_prefix = i[:-2]
        cur = gpd.read_file(os.path.join(dir_path, i, f"{file_prefix}.shp"))
        cur["year"] = year
        cur["month"] = month
        df = pd.concat([df, cur], ignore_index=True)

df.to_file(f"USDM_{YEAR}_All.geojson", driver="GeoJSON")